In [1]:
#!pip install pyspark

In [2]:
#import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

#initialize spark session
spark = SparkSession.builder.appName("SMS_Spam_Detection").getOrCreate()

# Phase 1: Data Ingestion

In [3]:
#load dataset
df = spark.read.csv("spam.csv", header=True, inferSchema=True)

#show schema
df.printSchema()

root
 |-- v1: string (nullable = true)
 |-- v2: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)



In [4]:
#show sample rows
df.show(5)

+----+--------------------+----+----+----+
|  v1|                  v2| _c2| _c3| _c4|
+----+--------------------+----+----+----+
| ham|Go until jurong p...|NULL|NULL|NULL|
| ham|Ok lar... Joking ...|NULL|NULL|NULL|
|spam|Free entry in 2 a...|NULL|NULL|NULL|
| ham|U dun say so earl...|NULL|NULL|NULL|
| ham|Nah I don't think...|NULL|NULL|NULL|
+----+--------------------+----+----+----+
only showing top 5 rows


## Phase 2: Data Cleaning

In [5]:
# ── 2a. Keep only the two useful columns and rename them ──
df = df.select(
    F.col("v1").alias("label"),
    F.col("v2").alias("message"),
)

print("\n── Step 1: Renamed columns ─────────────────")
df.show(5, truncate=70)


── Step 1: Renamed columns ─────────────────
+-----+----------------------------------------------------------------------+
|label|                                                               message|
+-----+----------------------------------------------------------------------+
|  ham|Go until jurong point, crazy.. Available only in bugis n great worl...|
|  ham|                                         Ok lar... Joking wif u oni...|
| spam|Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005....|
|  ham|                     U dun say so early hor... U c already then say...|
|  ham|         Nah I don't think he goes to usf, he lives around here though|
+-----+----------------------------------------------------------------------+
only showing top 5 rows


In [6]:
# ── 2b. Drop rows where label or message is null / empty ──
before = df.count()
df = df.filter(
    F.col("label").isNotNull() &
    F.col("message").isNotNull()   &
    (F.trim(F.col("message")) != "")
)
after = df.count()
print(f"\n── Step 2: Null / empty rows removed ──────")
print(f"   Before : {before:,}")
print(f"   After  : {after:,}")
print(f"   Dropped: {before - after:,}")


── Step 2: Null / empty rows removed ──────
   Before : 5,574
   After  : 5,573
   Dropped: 1


In [7]:
# ── 2c. Encode label: spam → 1, ham → 0 ─────
df = df.withColumn(
    "label",
    F.when(F.lower(F.col("label")) == "spam", 1)
     .when(F.lower(F.col("label")) == "ham",  0)
     .otherwise(None)                          # catches any unexpected value
    .cast(IntegerType())
)

# Drop rows with unexpected label values (if any)
unknown_labels = df.filter(F.col("label").isNull()).count()
if unknown_labels:
    print(f"\n   [WARN] {unknown_labels} rows had unrecognised labels — dropped.")
    df = df.filter(F.col("label").isNotNull())

df = df.select("message", "label")   # clean final schema

print("\n── Step 3: Label encoding (spam=1, ham=0) ──")
df.show(10, truncate=70)
print(f"\nClean dataset size: {df.count():,} rows")


   [WARN] 1 rows had unrecognised labels — dropped.

── Step 3: Label encoding (spam=1, ham=0) ──
+----------------------------------------------------------------------+-----+
|                                                               message|label|
+----------------------------------------------------------------------+-----+
|Go until jurong point, crazy.. Available only in bugis n great worl...|    0|
|                                         Ok lar... Joking wif u oni...|    0|
|Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005....|    1|
|                     U dun say so early hor... U c already then say...|    0|
|         Nah I don't think he goes to usf, he lives around here though|    0|
|FreeMsg Hey there darling it's been 3 week's now and no word back! ...|    1|
|Even my brother is not like to speak with me. They treat me like ai...|    0|
|As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vetta...|    0|
|WINNER!! As a valued network cu

# Phase 3: EDA

In [8]:
# ── 3a. Spam vs Ham counts ───────────────────
print("\n── 3a. Spam vs Ham distribution ────────────")
label_counts = (
    df.groupBy("label")
      .agg(F.count("*").alias("count"))
      .withColumn("class", F.when(F.col("label") == 1, "spam").otherwise("ham"))
      .orderBy("label")
)
label_counts.show()

counts = {row["label"]: row["count"] for row in label_counts.collect()}
spam_count = counts.get(1, 0)
ham_count  = counts.get(0, 0)
total = spam_count + ham_count
spam_pct   = spam_count / total * 100
ham_pct    = ham_count  / total * 100

print(f"   Ham  (0): {ham_count:>5,}  ({ham_pct:.1f}%)")
print(f"   Spam (1): {spam_count:>5,}  ({spam_pct:.1f}%)")
print(f"   Total   : {total:>5,}")


── 3a. Spam vs Ham distribution ────────────
+-----+-----+-----+
|label|count|class|
+-----+-----+-----+
|    0| 4825|  ham|
|    1|  747| spam|
+-----+-----+-----+

   Ham  (0): 4,825  (86.6%)
   Spam (1):   747  (13.4%)
   Total   : 5,572


In [9]:
# ── 3b. Message length distribution ─────────
print("\n── 3b. Message length stats (characters) ──")
df_len = df.withColumn("msg_length", F.length(F.col("message")))

df_len.groupBy("label").agg(
    F.round(F.mean("msg_length"),   1).alias("mean_len"),
    F.round(F.stddev("msg_length"), 1).alias("std_len"),
    F.min("msg_length").alias("min_len"),
    F.round(F.percentile_approx("msg_length", 0.25), 1).alias("q25"),
    F.round(F.percentile_approx("msg_length", 0.50), 1).alias("median"),
    F.round(F.percentile_approx("msg_length", 0.75), 1).alias("q75"),
    F.max("msg_length").alias("max_len"),
).withColumn(
    "class", F.when(F.col("label") == 1, "spam").otherwise("ham")
).select("class", "mean_len", "std_len", "min_len", "q25", "median", "q75", "max_len") \
 .orderBy("class") \
 .show()



── 3b. Message length stats (characters) ──
+-----+--------+-------+-------+---+------+---+-------+
|class|mean_len|std_len|min_len|q25|median|q75|max_len|
+-----+--------+-------+-------+---+------+---+-------+
|  ham|    71.1|   58.1|      2| 33|    52| 92|    910|
| spam|   138.5|   29.0|     13|132|   149|157|    223|
+-----+--------+-------+-------+---+------+---+-------+



In [10]:
# ── 3c. Length buckets (ASCII histogram) ─────
print("── 3c. Message length buckets ──────────────")
buckets = df_len.withColumn(
    "length_bucket",
    F.when(F.col("msg_length") <= 50,  "0–50")
     .when(F.col("msg_length") <= 100, "51–100")
     .when(F.col("msg_length") <= 150, "101–150")
     .when(F.col("msg_length") <= 200, "151–200")
     .otherwise("200+")
).groupBy("length_bucket", "label").count().orderBy("length_bucket", "label")
buckets.show()

── 3c. Message length buckets ──────────────
+-------------+-----+-----+
|length_bucket|label|count|
+-------------+-----+-----+
|         0–50|    0| 2326|
|         0–50|    1|   22|
|      101–150|    0|  655|
|      101–150|    1|  319|
|      151–200|    0|  307|
|      151–200|    1|  351|
|         200+|    0|  111|
|         200+|    1|    1|
|       51–100|    0| 1426|
|       51–100|    1|   54|
+-------------+-----+-----+



In [11]:
# ── 3d. Class imbalance ratio ────────────────
print("── 3d. Class imbalance summary ─────────────")
imbalance_ratio = ham_count / spam_count if spam_count else float("inf")
print(f"\n   Imbalance ratio (ham:spam) = {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print("   ⚠ Dataset shows class imbalance between spam and ham messages.")
    print("   Recommended handling approach within this task:")
    print("   • Use appropriate evaluation metrics (Precision, Recall, F1-score)")
    print("   • Avoid relying only on Accuracy for model evaluation")
    print("   • Use Logistic Regression with classWeight parameter (if needed in Spark)")
    print("   • Ensure stratified split or balanced train/test sampling if possible")

else:
    print("   ✓ Dataset is relatively balanced.")

── 3d. Class imbalance summary ─────────────

   Imbalance ratio (ham:spam) = 6.46:1
   ⚠ Dataset shows class imbalance between spam and ham messages.
   Recommended handling approach within this task:
   • Use appropriate evaluation metrics (Precision, Recall, F1-score)
   • Avoid relying only on Accuracy for model evaluation
   • Use Logistic Regression with classWeight parameter (if needed in Spark)
   • Ensure stratified split or balanced train/test sampling if possible


In [12]:
# from google.colab import drive
# drive.mount('/content/drive')

# Phase 4: Feature Engineering

In [13]:
#tokenize text
from pyspark.ml.feature import Tokenizer
tokenizer = Tokenizer(inputCol="message", outputCol="words")

#remove stopwords
from pyspark.ml.feature import StopWordsRemover
remover = StopWordsRemover(inputCol="words", outputCol="filtered")

#term frequency
from pyspark.ml.feature import HashingTF
tf = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=1000)

#idf
from pyspark.ml.feature import IDF
idf = IDF(inputCol="rawFeatures", outputCol="features")

#apply transformations step by step (for debugging)
df_words = tokenizer.transform(df)
df_filtered = remover.transform(df_words)
df_tf = tf.transform(df_filtered)
df_features = idf.fit(df_tf).transform(df_tf)

#preview features
df_features.select("features").show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
+---------------------------------------------

# Phase 5 & 6: Model + Pipeline

In [14]:
#logistic regression
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(featuresCol="features", labelCol="label")

#pipeline
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=[tokenizer, remover, tf, idf, lr])

#split data
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

#train model
model = pipeline.fit(train_data)

#predict
predictions = model.transform(test_data)

#show predictions
predictions.select("label", "prediction").show(5)

+-----+----------+
|label|prediction|
+-----+----------+
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
|    0|       0.0|
+-----+----------+
only showing top 5 rows


# Phase 7: Evaluation

In [15]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)

#precision
precision = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
).evaluate(predictions)

#recall
recall = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
).evaluate(predictions)

#f1
f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
).evaluate(predictions)

#print metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Accuracy: 0.9439775910364145
Precision: 0.946590943784807
Recall: 0.9439775910364145
F1-score: 0.9450620782484243


# Phase 8: Save & Load

In [16]:
from pyspark.ml import PipelineModel

#save trained pipeline model
model.save("/content/spam_pipeline_model")

#load the saved model
loaded_model = PipelineModel.load("/content/spam_pipeline_model")

# Phase 9: New Predictions

In [17]:
#new SMS messages
new_data = spark.createDataFrame([
    ("Free entry in a lottery! Claim now!",),
    ("Hey, are we still meeting for lunch today?",),
    ("Congratulations! You won a free ticket.",),
    ("Don't forget to submit the report by 5 PM.",)
], ["message"])

#predict using loaded model
new_predictions = loaded_model.transform(new_data)

#show results
new_predictions.select("message", "prediction").show()

+--------------------+----------+
|             message|prediction|
+--------------------+----------+
|Free entry in a l...|       1.0|
|Hey, are we still...|       0.0|
|Congratulations! ...|       0.0|
|Don't forget to s...|       0.0|
+--------------------+----------+

